In [ ]:
from langchain.messages import SystemMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from pydantic import Field, BaseModel
from pathlib import Path
from dotenv import load_dotenv
from typing import TypedDict, List, Literal, Any
import requests
import json
import os

In [ ]:
import google.generativeai as genai

d:\LearnXYZ\learnenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\ralxi\AppData\Local\Temp\ipykernel_8768\613638648.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [ ]:
load_dotenv()

True

In [ ]:
model = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash",
    api_key = os.getenv("GEMINI_API_KEY"),
    temperature = 0
)

In [ ]:
resp = model.invoke("Who is th president of india")
print(resp.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


The current President of India is **Droupadi Murmu**.

She assumed office on July 25, 2022, and is the 15th President of India. She is also the first tribal woman to hold the position.


In [ ]:
class Topic(BaseModel):
    name: str
    difficulty: Literal["High", "Medium", "Low"]
    definition: str
    Links: str
    articles: List

class Subtopic(BaseModel):
    subtopic_name: str
    difficulty: Literal["High", "Medium", "Low"]
    definition: str
    topics: List[Topic]

class MindMap(BaseModel):
    topic_name: str
    definition: str
    sutopics: List[Subtopic]

struct_model = model.with_structured_output(MindMap)

In [ ]:
resp = struct_model.invoke("generate a learning roadmap of Machine Learning")
print(resp)

topic_name='Machine Learning' definition='A field of artificial intelligence that enables systems to learn from data, identify patterns, and make decisions with minimal human intervention, without being explicitly programmed.' sutopics=[Subtopic(subtopic_name='Foundational Concepts & Mathematics', difficulty='High', definition='Essential mathematical and statistical concepts that underpin machine learning algorithms, along with fundamental programming skills.', topics=[Topic(name='Linear Algebra', difficulty='High', definition='Understanding vectors, matrices, eigenvalues, and their operations, crucial for data representation and algorithm mechanics.'), Topic(name='Calculus', difficulty='High', definition='Grasping derivatives, gradients, and optimization techniques (e.g., gradient descent) for training models.'), Topic(name='Probability & Statistics', difficulty='High', definition='Concepts like probability distributions, hypothesis testing, regression, and statistical inference for d

In [ ]:
prompt = f"""You are an expert learning-roadmap generator.

Your task is to create a structured learning roadmap for any topic provided by the user.

The roadmap must organize the topic from foundational concepts to advanced concepts and should help a learner understand what they should learn and in what order.

Follow these rules:

1. Identify the major subtopics required to understand the given topic.
2. Arrange the subtopics in a logical learning sequence, generally from foundational concepts to advanced concepts.
3. For every subtopic:
   - Provide a clear subtopic name.
   - Assign a difficulty level: "Low", "Medium", or "High".
   - Give a moderate amount of description about the topic
   - Identify the important concepts/topics that should be learned within that subtopic.
4. For every topic inside a subtopic:
   - Provide a concise and meaningful topic name.
   - Give a moderate amount of description about the topic
   - Assign a difficulty level: "Low", "Medium", or "High".
   - generate a youtube video link for the topic (give only the links for the youtube videos that exists don't try to make up links else write focus on articles)
   - 2-3 articles for the topic
5. Include prerequisite concepts before concepts that depend on them.
6. Do not include unnecessary or highly specialized concepts unless they are important for understanding the topic.
7. The roadmap should be comprehensive enough for a learner to progress from beginner to advanced level.
8. Avoid duplicate topics.
9. Difficulty should represent the relative complexity of learning the concept, not its importance.
10. The number of subtopics and topics should depend on the complexity of the input topic. Do not use a fixed number.
11. Return only the structured output defined by the provided schema. Do not return explanations, Markdown, comments, or additional text.

The input will be a topic that the user wants to learn.

return only in json format

not any other format

Generate the learning roadmap for that topic."""

In [ ]:
sysquery = SystemMessage(content=prompt)
humquery = HumanMessage(content="Machine Learning")
query = [sysquery,humquery]

response = struct_model.invoke(query)
roadmap = response.model_dump()

In [ ]:
print(roadmap)

{'topic_name': 'Machine Learning', 'definition': 'Machine Learning (ML) is a subset of Artificial Intelligence (AI) that enables systems to learn from data, identify patterns, and make decisions with minimal human intervention. It involves developing algorithms that can parse data, learn from it, and then make predictions or decisions.', 'sutopics': [{'subtopic_name': 'Foundational Math and Programming', 'difficulty': 'Low', 'definition': 'This section covers the essential mathematical and programming prerequisites necessary to understand and implement machine learning algorithms effectively. A strong grasp of these fundamentals will provide a solid base for more advanced topics.', 'topics': [{'name': 'Python Programming Basics', 'difficulty': 'Low', 'definition': 'Learn the syntax, data structures (lists, dictionaries, tuples, sets), control flow (if/else, loops), functions, and object-oriented programming concepts in Python, which is the most widely used language for ML.', 'Links': '

In [ ]:
class SubjectState(TypedDict):
    topic: str
    roadmap: dict[str, Any]

In [ ]:
def roadmapnode(state: SubjectState):
    sysquery = SystemMessage(content=prompt)
    humquery = HumanMessage(content=state["topic"])
    query = [sysquery,humquery]

    response = struct_model.invoke(query)
    roadmap = response.model_dump()
    return {"roadmap": roadmap}

def save_roadmap(state: SubjectState):

    path = Path("module.json")
    path.parent.mkdir(exist_ok=True)

    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        data = {}

    data[state["topic"]] = state["roadmap"]

    with open(path, "a", encoding="utf-8") as f:
        json.dump(data, f, indent=4)

    return {}


In [ ]:
graph = StateGraph(SubjectState)
graph.add_node("mindmap", roadmapnode)
graph.add_node("jsonsave", save_roadmap)
graph.add_edge(START, "mindmap")
graph.add_edge("mindmap", "jsonsave")
graph.add_edge("jsonsave", END)
workflow = graph.compile()

In [ ]:
initial_state = {"topic": "Machine Learning"}
final_state = workflow.invoke(initial_state)

In [ ]:
print(final_state["roadmap"])

{'topic_name': 'Machine Learning', 'definition': 'Machine Learning (ML) is a subset of Artificial Intelligence (AI) that enables systems to learn from data, identify patterns, and make decisions with minimal human intervention. It involves developing algorithms that can parse data, learn from it, and then make a prediction or decision.', 'sutopics': [{'subtopic_name': 'Foundational Math & Programming', 'difficulty': 'Low', 'definition': 'This section covers the essential mathematical and programming prerequisites necessary to understand and implement machine learning algorithms effectively. A strong foundation here will make subsequent topics much easier to grasp.', 'topics': [{'name': 'Python Programming Basics', 'difficulty': 'Low', 'definition': 'Learn the fundamentals of Python, including data types, control flow, functions, and basic data structures like lists, dictionaries, and tuples. This is the primary language for most ML development.', 'Links': 'https://www.youtube.com/watch